# Retail Shoplifting Detection via Pose-Based Temporal Analysis

This study presents a skeleton-based approach to automated shoplifting detection from CCTV footage,
using YOLO26s-pose for 17-keypoint body pose estimation and ByteTrack for multi-person tracking.
We extract 85 hierarchically organized skeleton features across four tiers (base keypoints,
velocity, spatial relationships, and advanced biomechanical features) and classify temporal
sequences using a hybrid LSTM-XGBoost architecture. Evaluated via 5-fold grouped cross-validation
on 358 balanced video clips with strict anti-leakage measures (GroupKFold by source video,
training-only scaling and augmentation), the system achieves a best F1 of 0.8187 and
AUC of 0.8133 using C_spatial features with LSTM. A systematic ablation study
across all four feature tiers and four baseline models reveals that spatial relationship features
provide the most discriminative signal for shoplifting detection.

## 1. Feature Extraction Pipeline

YOLO26s-pose detects 17 body keypoints per person per frame, while ByteTrack maintains consistent
person identities across frames. For each tracked individual, we extract 85 features organized
in four hierarchical tiers:

- **Tier A (Base, 36 features):** 17 keypoints x 2 coordinates (locally normalized to bounding box) + 2 cosine elbow angles
- **Tier B (+ Velocity, 72 features):** Tier A + frame-to-frame deltas of all 36 base features
- **Tier C (+ Spatial, 77 features):** Tier B + wrist-to-hip distances, hand height relative to shoulders, cross-body reach flags
- **Tier D (Full, 85 features):** Tier C + trunk angle, shoulder-hip ratio, head orientation, knee angles, wrist symmetry, bbox area change

This skeleton-based approach is privacy-preserving (no raw video stored), lighting-invariant,
and computationally efficient compared to full-frame video classification.

In [ ]:
"""
Phase 1: Feature Extraction
YOLO26s-pose + ByteTrack -> 85 skeleton features per frame per tracked person
"""

import os
import shutil
import numpy as np
import cv2
import yaml
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

# -- Paths -------------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\malho\Desktop\claudeagent\RETAILPROJECT")
CLASS_DIRS = {
    0: BASE_DIR / "data" / "Class_0_Normal",
    1: BASE_DIR / "data" / "Class_1_Shoplifting",
}
FEATURE_DIR = BASE_DIR / "v10_Features"
MAX_FRAMES = 30
MIN_FRAMES = 15

# -- Clean output directory --------------------------------------------------
if FEATURE_DIR.exists():
    shutil.rmtree(FEATURE_DIR)
for label in [0, 1]:
    (FEATURE_DIR / str(label)).mkdir(parents=True, exist_ok=True)

# -- Write ByteTrack config --------------------------------------------------
tracker_cfg = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.5,
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.6,
    "track_buffer": 30,
    "match_thresh": 0.8,
    "fuse_score": True,
    "gmc_method": "none",
}
tracker_path = BASE_DIR / "custom_tracker.yaml"
with open(tracker_path, "w") as f:
    yaml.dump(tracker_cfg, f)

# -- Load YOLO model ---------------------------------------------------------
model = YOLO("yolo26s-pose.pt")

# -- Helper functions ---------------------------------------------------------

def cosine_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    """Cosine angle at point b between points a-b-c, returned in [0,1]."""
    v1 = a - b
    v2 = c - b
    dot = np.dot(v1, v2)
    norm = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8
    cos_val = np.clip(dot / norm, -1.0, 1.0)
    return float(np.arccos(cos_val) / np.pi)


def extract_frame_features(
    kpts_xyn: np.ndarray,
    bbox_xyxy: np.ndarray,
    frame_w: int,
    frame_h: int,
) -> tuple:
    """
    Extract per-frame features (no velocity yet).
    Returns (49-dim feature array, bbox_area).
    Layout: base(36) + spatial(5) + advanced(8) = 49
    """
    kpts = kpts_xyn  # (17, 2) normalized to frame
    x1, y1, x2, y2 = bbox_xyxy
    bbox_w = x2 - x1
    bbox_h = y2 - y1
    bbox_area = bbox_w * bbox_h

    # -- GROUP A: BASE (36 features) --
    norm_kpts = np.zeros((17, 2), dtype=np.float32)
    for i in range(17):
        norm_x = (kpts[i, 0] - (x1 / frame_w)) / (bbox_w / frame_w + 1e-8)
        norm_y = (kpts[i, 1] - (y1 / frame_h)) / (bbox_h / frame_h + 1e-8)
        norm_kpts[i] = [norm_x, norm_y]

    base_coords = norm_kpts.flatten()  # 34 values
    left_elbow = cosine_angle(norm_kpts[5], norm_kpts[7], norm_kpts[9])
    right_elbow = cosine_angle(norm_kpts[6], norm_kpts[8], norm_kpts[10])
    group_a = np.concatenate([base_coords, [left_elbow, right_elbow]])  # 36

    # -- GROUP C: SPATIAL (5 features) --
    left_wrist_hip = np.linalg.norm(norm_kpts[9] - norm_kpts[11])
    right_wrist_hip = np.linalg.norm(norm_kpts[10] - norm_kpts[12])
    hand_height = (
        np.mean([norm_kpts[9, 1], norm_kpts[10, 1]])
        - np.mean([norm_kpts[5, 1], norm_kpts[6, 1]])
    )
    left_cross = 1.0 if norm_kpts[9, 0] > norm_kpts[6, 0] else 0.0
    right_cross = 1.0 if norm_kpts[10, 0] < norm_kpts[5, 0] else 0.0
    group_c = np.array([left_wrist_hip, right_wrist_hip, hand_height, left_cross, right_cross])

    # -- GROUP D: ADVANCED (8 features) --
    # Trunk angle
    shoulder_mid = (norm_kpts[5] + norm_kpts[6]) / 2
    hip_mid = (norm_kpts[11] + norm_kpts[12]) / 2
    trunk_vec = shoulder_mid - hip_mid
    vertical = np.array([0.0, -1.0])
    dot_t = np.dot(trunk_vec, vertical)
    norm_t = np.linalg.norm(trunk_vec) * np.linalg.norm(vertical) + 1e-8
    trunk_angle = float(np.arccos(np.clip(dot_t / norm_t, -1.0, 1.0)) / np.pi)

    # Shoulder-hip ratio
    shoulder_dist = np.linalg.norm(norm_kpts[5] - norm_kpts[6])
    hip_dist = np.linalg.norm(norm_kpts[11] - norm_kpts[12])
    sh_ratio = shoulder_dist / (hip_dist + 1e-6)

    # Head orientation
    head_x = norm_kpts[0, 0] - np.mean([norm_kpts[1, 0], norm_kpts[2, 0]])
    head_y = norm_kpts[0, 1] - np.mean([norm_kpts[1, 1], norm_kpts[2, 1]])

    # Knee angles
    left_knee = cosine_angle(norm_kpts[11], norm_kpts[13], norm_kpts[15])
    right_knee = cosine_angle(norm_kpts[12], norm_kpts[14], norm_kpts[16])

    # Placeholders for velocity-dependent features
    wrist_sym = 0.0
    bbox_change = 0.0

    group_d = np.array([
        trunk_angle, sh_ratio, head_x, head_y,
        left_knee, right_knee, wrist_sym, bbox_change,
    ])

    frame_feats = np.concatenate([group_a, group_c, group_d])  # 49
    return frame_feats, bbox_area


def build_full_features(frame_list: list, bbox_areas: list) -> np.ndarray:
    """
    Assemble 85-dim vectors: base(36) + velocity(36) + spatial(5) + advanced(8).
    Computes velocity, wrist symmetry, and bbox area change.
    """
    n_frames = len(frame_list)
    frames = np.array(frame_list)  # (N, 49)

    base = frames[:, :36]
    spatial = frames[:, 36:41]
    advanced = frames[:, 41:49].copy()

    # GROUP B: VELOCITY
    velocity = np.zeros_like(base)
    velocity[1:] = base[1:] - base[:-1]

    # Fix wrist symmetry (advanced index 6)
    # Left wrist = kpt9 -> base indices 18,19; right wrist = kpt10 -> 20,21
    for t in range(n_frames):
        left_wrist_vel = np.linalg.norm(velocity[t, 18:20])
        right_wrist_vel = np.linalg.norm(velocity[t, 20:22])
        advanced[t, 6] = abs(left_wrist_vel - right_wrist_vel)

    # Fix bbox area change (advanced index 7)
    areas = np.array(bbox_areas)
    for t in range(1, n_frames):
        advanced[t, 7] = (areas[t] - areas[t - 1]) / (areas[t - 1] + 1e-6)

    # Assemble: base(36) + velocity(36) + spatial(5) + advanced(8) = 85
    full = np.concatenate([base, velocity, spatial, advanced], axis=1)
    return full


# -- Main extraction loop ----------------------------------------------------
stats = {0: {"tracks": 0, "lengths": [], "zero_clips": []},
         1: {"tracks": 0, "lengths": [], "zero_clips": []}}

for label, class_dir in CLASS_DIRS.items():
    clips = sorted([f for f in class_dir.iterdir() if f.suffix == ".mp4"])
    class_name = "Normal" if label == 0 else "Shoplifting"
    print(f"\nProcessing {class_name}: {len(clips)} clips")

    for clip_path in tqdm(clips, desc=class_name):
        cap = cv2.VideoCapture(str(clip_path))
        if not cap.isOpened():
            print(f"  WARNING: Could not open {clip_path.name}")
            continue

        frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # {track_id: [(features_49, bbox_area), ...]}
        track_data = {}

        frame_count = 0
        while frame_count < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1

            results = model.track(
                frame,
                persist=True,
                verbose=False,
                conf=0.3,
                tracker=str(tracker_path),
                classes=[0],
            )

            r = results[0]
            if r.boxes is None or r.boxes.id is None or r.keypoints is None:
                continue

            boxes = r.boxes
            keypoints = r.keypoints

            for i in range(len(boxes)):
                track_id = int(boxes.id[i].item())
                bbox = boxes.xyxy[i].cpu().numpy()
                kpts = keypoints.xyn[i].cpu().numpy()

                feats, area = extract_frame_features(kpts, bbox, frame_w, frame_h)

                if track_id not in track_data:
                    track_data[track_id] = []
                track_data[track_id].append((feats, area))

        cap.release()

        # Process each track
        clip_name = clip_path.stem
        valid_tracks = 0
        for tid, data_list in track_data.items():
            if len(data_list) < MIN_FRAMES:
                continue

            data_list = data_list[:MAX_FRAMES]
            frame_feats = [d[0] for d in data_list]
            bbox_areas = [d[1] for d in data_list]

            full_features = build_full_features(frame_feats, bbox_areas)

            out_path = FEATURE_DIR / str(label) / f"{clip_name}_id{tid}.npy"
            np.save(out_path, full_features)
            valid_tracks += 1
            stats[label]["lengths"].append(len(data_list))

        stats[label]["tracks"] += valid_tracks
        if valid_tracks == 0:
            stats[label]["zero_clips"].append(clip_name)

        # Reset tracker between clips
        model.predictor = None

# -- Summary ------------------------------------------------------------------
print("\n" + "=" * 60)
print("FEATURE EXTRACTION SUMMARY")
print("=" * 60)
for label in [0, 1]:
    name = "Normal" if label == 0 else "Shoplifting"
    s = stats[label]
    lengths = s["lengths"]
    print(f"\n{name} (class {label}):")
    print(f"  Total tracks saved: {s['tracks']}")
    if lengths:
        print(f"  Sequence lengths -- avg: {np.mean(lengths):.1f}, "
              f"min: {np.min(lengths)}, max: {np.max(lengths)}")
    print(f"  Clips with zero valid tracks ({len(s['zero_clips'])}): "
          f"{s['zero_clips'][:10]}{'...' if len(s['zero_clips']) > 10 else ''}")

for label in [0, 1]:
    saved = list((FEATURE_DIR / str(label)).glob("*.npy"))
    if saved:
        sample = np.load(saved[0])
        print(f"\nClass {label}: {len(saved)} .npy files, sample shape: {sample.shape}")

print("\nDone.")


## 2. Model Training

The classification pipeline uses a two-stage hybrid architecture:

1. **LSTM (64 hidden units, 2 layers):** Captures temporal patterns in skeleton sequences using
   `pack_padded_sequence` for variable-length inputs. Trained with Focal Loss (alpha=0.75, gamma=2.0)
   to handle class imbalance, Adam optimizer (lr=0.0005), and early stopping (patience=15).

2. **XGBoost head:** Trained on the 64-dimensional LSTM hidden state representations extracted
   from training data only, adding non-linear classification power.

**Anti-leakage measures:**
- GroupKFold (5 splits) by source video ensures no scene information leaks between folds
- StandardScaler fit on training fold only
- Skeleton augmentation (horizontal flip, Gaussian noise, temporal crop, joint dropout) applied only during training
- Threshold optimization per fold via precision-recall curve F1 maximization
- Clip-level prediction aggregation using MAX across tracks

In [ ]:
"""
Phase 2: Training Pipeline (Zero Leakage)
LSTM (64 hidden) -> XGBoost hybrid, 5-fold GroupKFold, 4 feature tiers
"""

import os
import re
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve,
)
from xgboost import XGBClassifier
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# -- Paths -------------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\malho\Desktop\claudeagent\RETAILPROJECT")
FEATURE_DIR = BASE_DIR / "v10_Features"
RESULTS_DIR = BASE_DIR / "v10_Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -- Feature tier definitions ------------------------------------------------
FEATURE_TIERS = {
    "A_base":     list(range(0, 36)),
    "B_velocity": list(range(0, 72)),
    "C_spatial":  list(range(0, 77)),
    "D_full":     list(range(0, 85)),
}

# -- Data loading ------------------------------------------------------------
def load_all_data() -> list[dict]:
    """Scan all .npy files and return metadata list."""
    records = []
    for label in [0, 1]:
        label_dir = FEATURE_DIR / str(label)
        for npy_path in sorted(label_dir.glob("*.npy")):
            fname = npy_path.stem  # e.g. Normal_001_f1_id1
            # Remove _id{N} suffix to get clip_name
            clip_name = re.sub(r"_id\d+$", "", fname)
            # source_group: drop the last _part after rsplit
            source_group = clip_name.rsplit("_", 1)[0]
            data = np.load(npy_path)
            records.append({
                "path": str(npy_path),
                "label": label,
                "clip_name": clip_name,
                "source_group": source_group,
                "seq_length": data.shape[0],
                "data": data,
            })
    return records


# -- Dataset -----------------------------------------------------------------
class SkeletonDataset(Dataset):
    def __init__(
        self,
        records: list[dict],
        feature_cols: list[int],
        scaler: StandardScaler,
        augment: bool = False,
    ):
        self.records = records
        self.feature_cols = feature_cols
        self.augment = augment
        self.scaler = scaler
        self.n_features = len(feature_cols)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        seq = rec["data"][:, self.feature_cols].copy()  # (T, F)

        # Scale
        seq = self.scaler.transform(seq)

        if self.augment:
            seq = self._augment(seq)

        label = rec["label"]
        return (
            torch.FloatTensor(seq),
            torch.FloatTensor([label]),
            seq.shape[0],
            rec["clip_name"],
        )

    def _augment(self, seq: np.ndarray) -> np.ndarray:
        seq = seq.copy()
        n_feat = seq.shape[1]

        # 1. Horizontal Flip (50% chance)
        if np.random.rand() < 0.5 and n_feat >= 36:
            # Flip base x-coordinates (even indices 0,2,...,32)
            for i in range(0, min(34, n_feat), 2):
                seq[:, i] = -seq[:, i]  # negate (already scaled, so negate)

            # Swap left/right keypoint pairs in base (indices within 0-33)
            swap_pairs_base = [
                (10, 12), (11, 13),  # kpt5<->6
                (14, 16), (15, 17),  # kpt7<->8
                (18, 20), (19, 21),  # kpt9<->10
                (22, 24), (23, 25),  # kpt11<->12
                (26, 28), (27, 29),  # kpt13<->14
                (30, 32), (31, 33),  # kpt15<->16
            ]
            for a, b in swap_pairs_base:
                if a < n_feat and b < n_feat:
                    seq[:, [a, b]] = seq[:, [b, a]]

            # Swap elbow angles (34<->35)
            if n_feat > 35:
                seq[:, [34, 35]] = seq[:, [35, 34]]

            # Velocity features (indices 36-71)
            if n_feat >= 72:
                for i in range(36, 70, 2):
                    seq[:, i] = -seq[:, i]  # negate velocity x
                swap_pairs_vel = [(36 + a, 36 + b) for a, b in
                    [(10, 12), (11, 13), (14, 16), (15, 17),
                     (18, 20), (19, 21), (22, 24), (23, 25),
                     (26, 28), (27, 29), (30, 32), (31, 33)]]
                for a, b in swap_pairs_vel:
                    if a < n_feat and b < n_feat:
                        seq[:, [a, b]] = seq[:, [b, a]]

            # Spatial features (72-76)
            if n_feat >= 77:
                seq[:, [72, 73]] = seq[:, [73, 72]]  # wrist-hip left<->right
                seq[:, [75, 76]] = seq[:, [76, 75]]  # cross-body flags

            # Advanced features (77-84)
            if n_feat >= 85:
                seq[:, 79] = -seq[:, 79]  # negate head orientation x
                seq[:, [81, 82]] = seq[:, [82, 81]]  # swap knee angles

        # 2. Gaussian Noise (always)
        seq += np.random.normal(0, 0.01, seq.shape)

        # 3. Temporal Crop (30% chance, only if seq_length > 18)
        if np.random.rand() < 0.3 and seq.shape[0] > 18:
            crop_len = np.random.randint(3, 6)
            max_start = seq.shape[0] - crop_len - 1
            if max_start > 1:
                start = np.random.randint(1, max_start)
                seq = np.concatenate([seq[:start], seq[start + crop_len:]], axis=0)

        # 4. Joint Dropout (20% chance)
        if np.random.rand() < 0.2 and n_feat >= 36:
            n_drop = np.random.randint(1, 3)
            kpts = np.random.choice(17, n_drop, replace=False)
            for k in kpts:
                # Zero base coords
                if 2 * k + 1 < min(34, n_feat):
                    seq[:, 2 * k] = 0
                    seq[:, 2 * k + 1] = 0
                # Zero velocity coords
                if n_feat >= 72 and 36 + 2 * k + 1 < n_feat:
                    seq[:, 36 + 2 * k] = 0
                    seq[:, 36 + 2 * k + 1] = 0

        return seq


def collate_fn(batch):
    """Pad sequences to max length in batch, sort by length descending."""
    batch.sort(key=lambda x: x[2], reverse=True)
    seqs, labels, lengths, clip_names = zip(*batch)

    max_len = max(lengths)
    n_feat = seqs[0].shape[1]

    padded = torch.zeros(len(seqs), max_len, n_feat)
    for i, seq in enumerate(seqs):
        padded[i, :seq.shape[0]] = seq

    labels = torch.cat(labels)
    lengths = torch.LongTensor(lengths)
    return padded, labels, lengths, list(clip_names)


# -- Model -------------------------------------------------------------------
class RetailLSTM_V10(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            dropout=0.3,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, lengths, return_hidden: bool = False):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        output, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(output, batch_first=True)

        # Get last valid hidden state for each sample
        idx = (lengths - 1).long().unsqueeze(1).unsqueeze(2)
        idx = idx.expand(-1, 1, output.size(2)).to(output.device)
        hidden = output.gather(1, idx).squeeze(1)  # (B, 64)

        if return_hidden:
            return hidden

        logits = self.fc(hidden).squeeze(1)
        return torch.sigmoid(logits)


# -- Loss --------------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy(inputs, targets, reduction="none")
        pt = targets * inputs + (1 - targets) * (1 - inputs)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()


# -- Threshold optimization --------------------------------------------------
def find_optimal_threshold(y_true: np.ndarray, y_scores: np.ndarray) -> tuple:
    """Find threshold that maximizes F1 on clip-level scores."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    best_f1 = f1_scores[best_idx]
    return float(best_thresh), float(best_f1)


# -- Clip-level aggregation (MAX) -------------------------------------------
def aggregate_clip_scores(clip_names: list, scores: np.ndarray, labels: np.ndarray):
    """Aggregate track-level scores to clip-level using MAX."""
    clip_scores = {}
    clip_labels = {}
    for name, score, label in zip(clip_names, scores, labels):
        if name not in clip_scores:
            clip_scores[name] = []
            clip_labels[name] = label
        clip_scores[name].append(score)

    clip_names_unique = sorted(clip_scores.keys())
    agg_scores = np.array([max(clip_scores[n]) for n in clip_names_unique])
    agg_labels = np.array([clip_labels[n] for n in clip_names_unique])
    return clip_names_unique, agg_scores, agg_labels


# -- Training ----------------------------------------------------------------
def train_one_fold(
    train_records: list[dict],
    val_records: list[dict],
    feature_cols: list[int],
    fold_num: int,
    tier_name: str,
) -> dict:
    """Train LSTM + XGBoost for one fold, return metrics and predictions."""
    n_features = len(feature_cols)

    # Fit scaler on training data only
    all_train_frames = np.concatenate(
        [r["data"][:, feature_cols] for r in train_records], axis=0
    )
    scaler = StandardScaler()
    scaler.fit(all_train_frames)

    # Datasets
    train_ds = SkeletonDataset(train_records, feature_cols, scaler, augment=True)
    val_ds = SkeletonDataset(val_records, feature_cols, scaler, augment=False)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    # Model
    model = RetailLSTM_V10(input_size=n_features).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )
    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    # Training loop
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    max_epochs = 150
    early_stop_patience = 15

    for epoch in range(max_epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_batches = 0
        for seqs, labels, lengths, _ in train_loader:
            seqs, labels, lengths = seqs.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
            optimizer.zero_grad()
            preds = model(seqs, lengths)
            loss = criterion(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            train_batches += 1

        # Validate
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for seqs, labels, lengths, _ in val_loader:
                seqs, labels, lengths = seqs.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                preds = model(seqs, lengths)
                loss = criterion(preds, labels)
                val_loss += loss.item()
                val_batches += 1

        avg_train = train_loss / max(train_batches, 1)
        avg_val = val_loss / max(val_batches, 1)
        scheduler.step(avg_val)

        if (epoch + 1) % 10 == 0:
            print(f"  [{tier_name} Fold {fold_num}] Epoch {epoch+1}: "
                  f"train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stop_patience:
                print(f"  [{tier_name} Fold {fold_num}] Early stopping at epoch {epoch+1}")
                break

    # Load best checkpoint
    model.load_state_dict(best_state)
    model.to(DEVICE)
    model.eval()

    # --- LSTM-only evaluation ---
    val_scores_lstm = []
    val_labels_all = []
    val_clips_all = []
    with torch.no_grad():
        for seqs, labels, lengths, clip_names in val_loader:
            seqs, lengths = seqs.to(DEVICE), lengths.to(DEVICE)
            preds = model(seqs, lengths)
            val_scores_lstm.extend(preds.cpu().numpy().tolist())
            val_labels_all.extend(labels.numpy().tolist())
            val_clips_all.extend(clip_names)

    val_scores_lstm = np.array(val_scores_lstm)
    val_labels_all = np.array(val_labels_all)

    # Clip-level aggregation for LSTM
    clip_names_u, clip_scores_lstm, clip_labels = aggregate_clip_scores(
        val_clips_all, val_scores_lstm, val_labels_all
    )
    thresh_lstm, _ = find_optimal_threshold(clip_labels, clip_scores_lstm)
    clip_preds_lstm = (clip_scores_lstm >= thresh_lstm).astype(int)

    lstm_metrics = {
        "accuracy": accuracy_score(clip_labels, clip_preds_lstm),
        "precision": precision_score(clip_labels, clip_preds_lstm, zero_division=0),
        "recall": recall_score(clip_labels, clip_preds_lstm, zero_division=0),
        "f1": f1_score(clip_labels, clip_preds_lstm, zero_division=0),
        "auc": roc_auc_score(clip_labels, clip_scores_lstm) if len(np.unique(clip_labels)) > 1 else 0.0,
        "threshold": thresh_lstm,
    }

    # --- XGBoost hybrid ---
    # Extract hidden states from TRAINING data (no augmentation)
    train_ds_no_aug = SkeletonDataset(train_records, feature_cols, scaler, augment=False)
    train_loader_no_aug = DataLoader(
        train_ds_no_aug, batch_size=64, shuffle=False, collate_fn=collate_fn
    )

    train_hiddens = []
    train_labels_xgb = []
    with torch.no_grad():
        for seqs, labels, lengths, _ in train_loader_no_aug:
            seqs, lengths = seqs.to(DEVICE), lengths.to(DEVICE)
            h = model(seqs, lengths, return_hidden=True)
            train_hiddens.append(h.cpu().numpy())
            train_labels_xgb.extend(labels.numpy().tolist())

    train_hiddens = np.concatenate(train_hiddens, axis=0)
    train_labels_xgb = np.array(train_labels_xgb)

    # Count for scale_pos_weight
    n_class0 = np.sum(train_labels_xgb == 0)
    n_class1 = np.sum(train_labels_xgb == 1)
    spw = n_class0 / max(n_class1, 1)

    xgb = XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=spw,
        eval_metric="logloss",
        verbosity=0,
        random_state=42,
    )
    xgb.fit(train_hiddens, train_labels_xgb)

    # Val hidden states
    val_hiddens = []
    with torch.no_grad():
        for seqs, labels, lengths, _ in val_loader:
            seqs, lengths = seqs.to(DEVICE), lengths.to(DEVICE)
            h = model(seqs, lengths, return_hidden=True)
            val_hiddens.append(h.cpu().numpy())

    val_hiddens = np.concatenate(val_hiddens, axis=0)
    val_scores_xgb = xgb.predict_proba(val_hiddens)[:, 1]

    # Clip-level aggregation for hybrid
    clip_names_u2, clip_scores_xgb, clip_labels2 = aggregate_clip_scores(
        val_clips_all, val_scores_xgb, val_labels_all
    )
    thresh_xgb, _ = find_optimal_threshold(clip_labels2, clip_scores_xgb)
    clip_preds_xgb = (clip_scores_xgb >= thresh_xgb).astype(int)

    hybrid_metrics = {
        "accuracy": accuracy_score(clip_labels2, clip_preds_xgb),
        "precision": precision_score(clip_labels2, clip_preds_xgb, zero_division=0),
        "recall": recall_score(clip_labels2, clip_preds_xgb, zero_division=0),
        "f1": f1_score(clip_labels2, clip_preds_xgb, zero_division=0),
        "auc": roc_auc_score(clip_labels2, clip_scores_xgb) if len(np.unique(clip_labels2)) > 1 else 0.0,
        "threshold": thresh_xgb,
    }

    # Per-clip results for error analysis
    clip_results = []
    for i, name in enumerate(clip_names_u2):
        clip_results.append({
            "clip_name": name,
            "true_label": int(clip_labels2[i]),
            "lstm_score": float(clip_scores_lstm[i]) if i < len(clip_scores_lstm) else 0.0,
            "hybrid_score": float(clip_scores_xgb[i]),
            "hybrid_pred": int(clip_preds_xgb[i]),
            "lstm_pred": int(clip_preds_lstm[i]) if i < len(clip_preds_lstm) else 0,
        })

    return {
        "lstm_metrics": lstm_metrics,
        "hybrid_metrics": hybrid_metrics,
        "clip_results": clip_results,
        "clip_scores_lstm": clip_scores_lstm.tolist(),
        "clip_scores_xgb": clip_scores_xgb.tolist(),
        "clip_labels": clip_labels2.tolist(),
        "clip_names": clip_names_u2,
        "scaler": scaler,
        "model_state": best_state,
        "xgb_model": xgb,
    }


# -- Main --------------------------------------------------------------------
def main():
    print("Loading data...")
    all_records = load_all_data()
    print(f"Loaded {len(all_records)} tracks")

    # Extract arrays for GroupKFold
    groups = np.array([r["source_group"] for r in all_records])
    labels = np.array([r["label"] for r in all_records])

    gkf = GroupKFold(n_splits=5)
    fold_indices = list(gkf.split(np.zeros(len(all_records)), labels, groups))

    # Save fold assignments
    with open(RESULTS_DIR / "fold_assignments.pkl", "wb") as f:
        pickle.dump(fold_indices, f)

    # Print fold composition
    print("\nFold composition:")
    for i, (train_idx, val_idx) in enumerate(fold_indices):
        train_groups = set(groups[train_idx])
        val_groups = set(groups[val_idx])
        print(f"  Fold {i+1}: Train = {len(train_groups)} groups ({len(train_idx)} tracks) | "
              f"Val = {len(val_groups)} groups ({len(val_idx)} tracks)")

    # Train all tiers x folds
    all_results = {}

    for tier_name, feature_cols in FEATURE_TIERS.items():
        print(f"\n{'='*60}")
        print(f"TIER: {tier_name} ({len(feature_cols)} features)")
        print(f"{'='*60}")

        all_results[tier_name] = {}

        for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
            print(f"\n--- Fold {fold_num}/5 ---")
            train_recs = [all_records[i] for i in train_idx]
            val_recs = [all_records[i] for i in val_idx]

            fold_result = train_one_fold(
                train_recs, val_recs, feature_cols, fold_num, tier_name
            )
            all_results[tier_name][fold_num] = fold_result

            print(f"  LSTM:   F1={fold_result['lstm_metrics']['f1']:.4f} "
                  f"AUC={fold_result['lstm_metrics']['auc']:.4f} "
                  f"(thresh={fold_result['lstm_metrics']['threshold']:.3f})")
            print(f"  Hybrid: F1={fold_result['hybrid_metrics']['f1']:.4f} "
                  f"AUC={fold_result['hybrid_metrics']['auc']:.4f} "
                  f"(thresh={fold_result['hybrid_metrics']['threshold']:.3f})")

    # Save best Tier B model (best fold by hybrid F1)
    tier_b = all_results["B_velocity"]
    best_fold = max(tier_b.keys(), key=lambda k: tier_b[k]["hybrid_metrics"]["f1"])
    best_result = tier_b[best_fold]

    torch.save(best_result["model_state"], RESULTS_DIR / "lstm_v10.pth")
    with open(RESULTS_DIR / "xgboost_v10.pkl", "wb") as f:
        pickle.dump(best_result["xgb_model"], f)
    with open(RESULTS_DIR / "scaler_v10.pkl", "wb") as f:
        pickle.dump(best_result["scaler"], f)

    # Save all results (strip non-serializable objects for the main pickle)
    results_to_save = {}
    for tier in all_results:
        results_to_save[tier] = {}
        for fold in all_results[tier]:
            r = all_results[tier][fold]
            results_to_save[tier][fold] = {
                "lstm_metrics": r["lstm_metrics"],
                "hybrid_metrics": r["hybrid_metrics"],
                "clip_results": r["clip_results"],
                "clip_scores_lstm": r["clip_scores_lstm"],
                "clip_scores_xgb": r["clip_scores_xgb"],
                "clip_labels": r["clip_labels"],
                "clip_names": r["clip_names"],
            }

    with open(RESULTS_DIR / "all_results.pkl", "wb") as f:
        pickle.dump(results_to_save, f)

    # Print summary
    print("\n" + "=" * 60)
    print("TRAINING SUMMARY")
    print("=" * 60)
    for tier_name in FEATURE_TIERS:
        tier_res = all_results[tier_name]
        lstm_f1s = [tier_res[f]["lstm_metrics"]["f1"] for f in tier_res]
        lstm_aucs = [tier_res[f]["lstm_metrics"]["auc"] for f in tier_res]
        hyb_f1s = [tier_res[f]["hybrid_metrics"]["f1"] for f in tier_res]
        hyb_aucs = [tier_res[f]["hybrid_metrics"]["auc"] for f in tier_res]

        n_feat = len(FEATURE_TIERS[tier_name])
        print(f"\nTIER {tier_name} ({n_feat} features):")
        print(f"  LSTM:   F1 = {np.mean(lstm_f1s):.4f} +/- {np.std(lstm_f1s):.4f} | "
              f"AUC = {np.mean(lstm_aucs):.4f} +/- {np.std(lstm_aucs):.4f}")
        print(f"  Hybrid: F1 = {np.mean(hyb_f1s):.4f} +/- {np.std(hyb_f1s):.4f} | "
              f"AUC = {np.mean(hyb_aucs):.4f} +/- {np.std(hyb_aucs):.4f}")

    print(f"\nBest Tier B model saved from fold {best_fold}")
    print("All results saved to v10_Results/all_results.pkl")
    print("Done.")


if __name__ == "__main__":
    main()


## 3. Baselines and Ablation Study

To establish fair comparisons, we train four baseline models on the same GroupKFold splits
using flattened Tier B features (30 frames x 72 features = 2,160-dim input vectors):

1. **Majority Class** -- lower bound
2. **Logistic Regression** (balanced class weights)
3. **Random Forest** (200 trees, balanced)
4. **XGBoost** (150 estimators, scale_pos_weight)

All models use the same threshold optimization and clip-level MAX aggregation for fair comparison.
The ablation study tests whether temporal modeling (LSTM vs flat baselines), the hybrid architecture
(LSTM+XGB vs LSTM-only), and each feature tier contribute meaningful performance gains.

In [ ]:
"""
Phase 3: Baselines + Full Ablation Table
Trains 4 baseline models on flattened Tier B features, combines with Phase 2 results.
"""

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve,
)
from xgboost import XGBClassifier
import re
import warnings
warnings.filterwarnings("ignore")

# -- Paths -------------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\malho\Desktop\claudeagent\RETAILPROJECT")
FEATURE_DIR = BASE_DIR / "v10_Features"
RESULTS_DIR = BASE_DIR / "v10_Results"

# -- Load Phase 2 artifacts --------------------------------------------------
print("Loading Phase 2 results...")
with open(RESULTS_DIR / "fold_assignments.pkl", "rb") as f:
    fold_indices = pickle.load(f)

with open(RESULTS_DIR / "all_results.pkl", "rb") as f:
    phase2_results = pickle.load(f)

# -- Feature tier for baselines (Tier B = 72 features) ----------------------
TIER_B_COLS = list(range(0, 72))
PAD_LENGTH = 30  # Pad/truncate to 30 frames, flatten -> 30*72=2160

# -- Load all data -----------------------------------------------------------
def load_all_data() -> list[dict]:
    records = []
    for label in [0, 1]:
        label_dir = FEATURE_DIR / str(label)
        for npy_path in sorted(label_dir.glob("*.npy")):
            fname = npy_path.stem
            clip_name = re.sub(r"_id\d+$", "", fname)
            source_group = clip_name.rsplit("_", 1)[0]
            data = np.load(npy_path)
            records.append({
                "path": str(npy_path),
                "label": label,
                "clip_name": clip_name,
                "source_group": source_group,
                "seq_length": data.shape[0],
                "data": data,
            })
    return records


def flatten_features(records: list[dict], feature_cols: list[int], pad_len: int) -> np.ndarray:
    """Pad each track to pad_len frames, slice feature cols, flatten."""
    n_feat = len(feature_cols)
    X = np.zeros((len(records), pad_len * n_feat), dtype=np.float32)
    for i, rec in enumerate(records):
        seq = rec["data"][:, feature_cols]  # (T, F)
        T = min(seq.shape[0], pad_len)
        padded = np.zeros((pad_len, n_feat), dtype=np.float32)
        padded[:T] = seq[:T]
        X[i] = padded.flatten()
    return X


def find_optimal_threshold(y_true: np.ndarray, y_scores: np.ndarray) -> tuple:
    """Find threshold that maximizes F1."""
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.0
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return float(best_thresh), float(f1_scores[best_idx])


def aggregate_clip_scores(clip_names: list, scores: np.ndarray, labels: np.ndarray):
    """MAX aggregation at clip level."""
    clip_scores = {}
    clip_labels = {}
    for name, score, label in zip(clip_names, scores, labels):
        if name not in clip_scores:
            clip_scores[name] = []
            clip_labels[name] = label
        clip_scores[name].append(score)
    names = sorted(clip_scores.keys())
    agg_scores = np.array([max(clip_scores[n]) for n in names])
    agg_labels = np.array([clip_labels[n] for n in names])
    return names, agg_scores, agg_labels


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_scores: np.ndarray) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_scores) if len(np.unique(y_true)) > 1 else 0.0,
    }


# -- Main --------------------------------------------------------------------
print("Loading data...")
all_records = load_all_data()
print(f"Loaded {len(all_records)} tracks")

labels_all = np.array([r["label"] for r in all_records])
clip_names_all = [r["clip_name"] for r in all_records]

# Store baseline results: {model_name: {fold: metrics}}
baseline_results = {
    "Majority Class": {},
    "Logistic Regression": {},
    "Random Forest": {},
    "XGBoost (flat)": {},
}

for fold_num, (train_idx, val_idx) in enumerate(fold_indices, 1):
    print(f"\n--- Fold {fold_num}/5 ---")
    train_recs = [all_records[i] for i in train_idx]
    val_recs = [all_records[i] for i in val_idx]

    train_labels = np.array([r["label"] for r in train_recs])
    val_labels = np.array([r["label"] for r in val_recs])
    train_clips = [r["clip_name"] for r in train_recs]
    val_clips = [r["clip_name"] for r in val_recs]

    # Flatten features
    X_train = flatten_features(train_recs, TIER_B_COLS, PAD_LENGTH)
    X_val = flatten_features(val_recs, TIER_B_COLS, PAD_LENGTH)

    # Scale
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # --- 1. Majority Class ---
    majority_class = int(np.argmax(np.bincount(train_labels)))
    maj_preds = np.full(len(val_labels), majority_class)
    maj_scores = np.full(len(val_labels), float(majority_class))
    # Clip aggregation
    cn, cs, cl = aggregate_clip_scores(val_clips, maj_scores, val_labels)
    cp = np.full(len(cl), majority_class)
    baseline_results["Majority Class"][fold_num] = compute_metrics(cl, cp, cs)
    print(f"  Majority Class: F1={baseline_results['Majority Class'][fold_num]['f1']:.4f}")

    # --- 2. Logistic Regression ---
    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(X_train, train_labels)
    lr_scores = lr.predict_proba(X_val)[:, 1]
    cn, cs, cl = aggregate_clip_scores(val_clips, lr_scores, val_labels)
    thresh, _ = find_optimal_threshold(cl, cs)
    cp = (cs >= thresh).astype(int)
    baseline_results["Logistic Regression"][fold_num] = compute_metrics(cl, cp, cs)
    print(f"  Logistic Regression: F1={baseline_results['Logistic Regression'][fold_num]['f1']:.4f} (thresh={thresh:.3f})")

    # --- 3. Random Forest ---
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    rf.fit(X_train, train_labels)
    rf_scores = rf.predict_proba(X_val)[:, 1]
    cn, cs, cl = aggregate_clip_scores(val_clips, rf_scores, val_labels)
    thresh, _ = find_optimal_threshold(cl, cs)
    cp = (cs >= thresh).astype(int)
    baseline_results["Random Forest"][fold_num] = compute_metrics(cl, cp, cs)
    print(f"  Random Forest: F1={baseline_results['Random Forest'][fold_num]['f1']:.4f} (thresh={thresh:.3f})")

    # --- 4. XGBoost Direct ---
    n0 = np.sum(train_labels == 0)
    n1 = np.sum(train_labels == 1)
    xgb = XGBClassifier(
        n_estimators=150, max_depth=5, scale_pos_weight=n0 / max(n1, 1),
        eval_metric="logloss", verbosity=0, random_state=42,
    )
    xgb.fit(X_train, train_labels)
    xgb_scores = xgb.predict_proba(X_val)[:, 1]
    cn, cs, cl = aggregate_clip_scores(val_clips, xgb_scores, val_labels)
    thresh, _ = find_optimal_threshold(cl, cs)
    cp = (cs >= thresh).astype(int)
    baseline_results["XGBoost (flat)"][fold_num] = compute_metrics(cl, cp, cs)
    print(f"  XGBoost (flat): F1={baseline_results['XGBoost (flat)'][fold_num]['f1']:.4f} (thresh={thresh:.3f})")


# -- Build ablation table ---------------------------------------------------
print("\n" + "=" * 60)
print("BUILDING ABLATION TABLE")
print("=" * 60)

TIER_NAMES = {
    "A_base": ("A: 36 feat", 36),
    "B_velocity": ("B: 72 feat", 72),
    "C_spatial": ("C: 77 feat", 77),
    "D_full": ("D: 85 feat", 85),
}

rows = []

# Baseline rows (1-4)
for model_name in ["Majority Class", "Logistic Regression", "Random Forest", "XGBoost (flat)"]:
    fold_metrics = baseline_results[model_name]
    metrics_agg = {}
    for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
        vals = [fold_metrics[f][metric] for f in fold_metrics]
        metrics_agg[metric] = f"{np.mean(vals):.4f} +/- {np.std(vals):.4f}"
    rows.append({
        "Model": model_name,
        "Features": 72,
        **metrics_agg,
    })

# LSTM and Hybrid rows (5-12)
for tier_key, (tier_label, n_feat) in TIER_NAMES.items():
    tier_res = phase2_results[tier_key]

    for model_type, metrics_key in [("LSTM Only", "lstm_metrics"), ("LSTM+XGB", "hybrid_metrics")]:
        metrics_agg = {}
        for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
            vals = [tier_res[f][metrics_key][metric] for f in tier_res]
            metrics_agg[metric] = f"{np.mean(vals):.4f} +/- {np.std(vals):.4f}"
        rows.append({
            "Model": f"{model_type} (Tier {tier_label})",
            "Features": n_feat,
            **metrics_agg,
        })

# Create DataFrame
df = pd.DataFrame(rows)
df.index = range(1, len(df) + 1)
df.index.name = "#"

# Save
df.to_csv(RESULTS_DIR / "ablation_table.csv")

# Save baseline results for Phase 4
with open(RESULTS_DIR / "baseline_results.pkl", "wb") as f:
    pickle.dump(baseline_results, f)

# Print table
print("\n" + df.to_string())

# -- Interpretation ----------------------------------------------------------
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

# Extract mean F1 for comparison
def get_mean_f1(tier_key: str, model_type: str) -> float:
    metrics_key = "lstm_metrics" if model_type == "lstm" else "hybrid_metrics"
    vals = [phase2_results[tier_key][f][metrics_key]["f1"] for f in phase2_results[tier_key]]
    return np.mean(vals)

def get_baseline_mean_f1(model_name: str) -> float:
    vals = [baseline_results[model_name][f]["f1"] for f in baseline_results[model_name]]
    return np.mean(vals)

best_baseline_f1 = max(get_baseline_mean_f1(m) for m in baseline_results)
best_baseline_name = max(baseline_results.keys(), key=lambda m: get_baseline_mean_f1(m))

tier_a_lstm = get_mean_f1("A_base", "lstm")
tier_b_lstm = get_mean_f1("B_velocity", "lstm")
tier_c_lstm = get_mean_f1("C_spatial", "lstm")
tier_d_lstm = get_mean_f1("D_full", "lstm")

tier_a_hyb = get_mean_f1("A_base", "hybrid")
tier_b_hyb = get_mean_f1("B_velocity", "hybrid")
tier_c_hyb = get_mean_f1("C_spatial", "hybrid")
tier_d_hyb = get_mean_f1("D_full", "hybrid")

all_f1s = {
    "A_lstm": tier_a_lstm, "A_hyb": tier_a_hyb,
    "B_lstm": tier_b_lstm, "B_hyb": tier_b_hyb,
    "C_lstm": tier_c_lstm, "C_hyb": tier_c_hyb,
    "D_lstm": tier_d_lstm, "D_hyb": tier_d_hyb,
}
best_overall = max(all_f1s, key=all_f1s.get)
best_f1 = all_f1s[best_overall]

print(f"\n1. Best performing model: {best_overall} with F1={best_f1:.4f}")
print(f"2. Best baseline: {best_baseline_name} with F1={best_baseline_f1:.4f}")
print(f"   -> LSTM/Hybrid beats best baseline by {(best_f1 - best_baseline_f1)*100:.1f}% F1")
print(f"3. Velocity features (B vs A): LSTM {tier_b_lstm:.4f} vs {tier_a_lstm:.4f} "
      f"({'helped' if tier_b_lstm > tier_a_lstm else 'did not help'})")
print(f"4. Spatial features (C vs B): LSTM {tier_c_lstm:.4f} vs {tier_b_lstm:.4f} "
      f"({'helped' if tier_c_lstm > tier_b_lstm else 'did not help'})")
print(f"5. Full features (D vs C): LSTM {tier_d_lstm:.4f} vs {tier_c_lstm:.4f} "
      f"({'overfitting likely' if tier_d_lstm < tier_c_lstm else 'still helping'})")
print(f"6. Hybrid vs LSTM-only (best tier): "
      f"{'Hybrid better' if max(tier_a_hyb, tier_b_hyb, tier_c_hyb, tier_d_hyb) > max(tier_a_lstm, tier_b_lstm, tier_c_lstm, tier_d_lstm) else 'LSTM-only better or comparable'}")

print("\nDone.")


## 4. Results and Visualization

In [ ]:
"""
Phase 4: Visualizations + Error Analysis
Generates 7 plots and error analysis report.
"""

import pickle
import re
import numpy as np
import pandas as pd
import matplotlib
# matplotlib inline for notebook
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix

# -- Paths -------------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\malho\Desktop\claudeagent\RETAILPROJECT")
FEATURE_DIR = BASE_DIR / "v10_Features"
RESULTS_DIR = BASE_DIR / "v10_Results"

plt.style.use("seaborn-v0_8-whitegrid")

# -- Load data ---------------------------------------------------------------
print("Loading results...")
with open(RESULTS_DIR / "all_results.pkl", "rb") as f:
    phase2_results = pickle.load(f)

with open(RESULTS_DIR / "baseline_results.pkl", "rb") as f:
    baseline_results = pickle.load(f)

ablation_df = pd.read_csv(RESULTS_DIR / "ablation_table.csv", index_col=0)

# -- Load feature files for data overview ------------------------------------
def load_feature_stats():
    lengths = {0: [], 1: []}
    counts = {0: 0, 1: 0}
    for label in [0, 1]:
        label_dir = FEATURE_DIR / str(label)
        for npy_path in label_dir.glob("*.npy"):
            data = np.load(npy_path)
            lengths[label].append(data.shape[0])
            counts[label] += 1
    return counts, lengths

counts, lengths = load_feature_stats()

# ============================================================================
# PLOT 1: Data Overview
# ============================================================================
print("Plot 1: Data Overview...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: class distribution
classes = ["Normal", "Shoplifting"]
class_counts = [counts[0], counts[1]]
bars = ax1.bar(classes, class_counts, color=["#4CAF50", "#F44336"], edgecolor="black")
for bar, count in zip(bars, class_counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             str(count), ha="center", fontweight="bold")
ax1.set_ylabel("Number of Tracks")
ax1.set_title("Class Distribution")

# Right: sequence length histogram
ax2.hist(lengths[0], bins=15, alpha=0.6, label="Normal", color="#4CAF50", edgecolor="black")
ax2.hist(lengths[1], bins=15, alpha=0.6, label="Shoplifting", color="#F44336", edgecolor="black")
ax2.set_xlabel("Sequence Length (frames)")
ax2.set_ylabel("Count")
ax2.set_title("Sequence Length Distribution")
ax2.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot1_data_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 2: Feature Tier Comparison (KEY PLOT)
# ============================================================================
print("Plot 2: Feature Tier Comparison...")
fig, ax = plt.subplots(figsize=(10, 5))

tier_names = ["A_base", "B_velocity", "C_spatial", "D_full"]
tier_labels = ["Tier A\n(36 feat)", "Tier B\n(72 feat)", "Tier C\n(77 feat)", "Tier D\n(85 feat)"]

lstm_f1_means = []
lstm_f1_stds = []
hyb_f1_means = []
hyb_f1_stds = []

for tier in tier_names:
    lstm_vals = [phase2_results[tier][f]["lstm_metrics"]["f1"] for f in phase2_results[tier]]
    hyb_vals = [phase2_results[tier][f]["hybrid_metrics"]["f1"] for f in phase2_results[tier]]
    lstm_f1_means.append(np.mean(lstm_vals))
    lstm_f1_stds.append(np.std(lstm_vals))
    hyb_f1_means.append(np.mean(hyb_vals))
    hyb_f1_stds.append(np.std(hyb_vals))

x = np.arange(len(tier_labels))
width = 0.35

bars1 = ax.bar(x - width / 2, lstm_f1_means, width, yerr=lstm_f1_stds,
               label="LSTM Only", color="#2196F3", capsize=5, edgecolor="black")
bars2 = ax.bar(x + width / 2, hyb_f1_means, width, yerr=hyb_f1_stds,
               label="LSTM+XGB Hybrid", color="#4CAF50", capsize=5, edgecolor="black")

ax.set_ylabel("F1 Score")
ax.set_title("Feature Tier Comparison: Which Features Matter?")
ax.set_xticks(x)
ax.set_xticklabels(tier_labels)
ax.legend()
ax.set_ylim(0.5, 1.0)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01,
                f"{height:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot2_tier_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 3: Full Model Comparison
# ============================================================================
print("Plot 3: Full Model Comparison...")
fig, ax = plt.subplots(figsize=(12, 5))

# Parse F1 means from ablation table
model_names = ablation_df["Model"].tolist()
f1_means = []
f1_stds = []
for f1_str in ablation_df["f1"].tolist():
    parts = f1_str.split(" +/- ")
    f1_means.append(float(parts[0]))
    f1_stds.append(float(parts[1]))

# Sort descending
sorted_idx = np.argsort(f1_means)[::-1]
sorted_names = [model_names[i] for i in sorted_idx]
sorted_means = [f1_means[i] for i in sorted_idx]
sorted_stds = [f1_stds[i] for i in sorted_idx]

# Color by type
colors = []
for name in sorted_names:
    if "LSTM+XGB" in name:
        colors.append("#4CAF50")
    elif "LSTM" in name:
        colors.append("#2196F3")
    else:
        colors.append("#9E9E9E")

bars = ax.barh(range(len(sorted_names)), sorted_means, xerr=sorted_stds,
               color=colors, capsize=3, edgecolor="black")
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=8)
ax.set_xlabel("F1 Score")
ax.set_title("All Models Ranked by F1 Score")
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#9E9E9E", edgecolor="black", label="Baseline"),
    Patch(facecolor="#2196F3", edgecolor="black", label="LSTM Only"),
    Patch(facecolor="#4CAF50", edgecolor="black", label="LSTM+XGB Hybrid"),
]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot3_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 4: ROC Curves
# ============================================================================
print("Plot 4: ROC Curves...")
fig, ax = plt.subplots(figsize=(8, 8))

# Find best fold for Tier B hybrid (by AUC)
tier_b = phase2_results["B_velocity"]
best_fold_b = max(tier_b.keys(), key=lambda k: tier_b[k]["hybrid_metrics"]["auc"])

# Find best overall tier by mean hybrid AUC
tier_aucs = {}
for tier in tier_names:
    vals = [phase2_results[tier][f]["hybrid_metrics"]["auc"] for f in phase2_results[tier]]
    tier_aucs[tier] = np.mean(vals)
best_tier = max(tier_aucs, key=tier_aucs.get)

# Plot ROC curves
curves_to_plot = [
    ("B_velocity", best_fold_b, "LSTM Only (Tier B)", "lstm", "#2196F3", "-"),
    ("B_velocity", best_fold_b, "Hybrid (Tier B)", "hybrid", "#4CAF50", "-"),
]
if best_tier != "B_velocity":
    best_fold_other = max(phase2_results[best_tier].keys(),
                          key=lambda k: phase2_results[best_tier][k]["hybrid_metrics"]["auc"])
    curves_to_plot.append(
        (best_tier, best_fold_other, f"Hybrid ({best_tier})", "hybrid", "#FF9800", "--")
    )

for tier, fold, label, model_type, color, ls in curves_to_plot:
    r = phase2_results[tier][fold]
    scores_key = "clip_scores_lstm" if model_type == "lstm" else "clip_scores_xgb"
    y_true = np.array(r["clip_labels"])
    y_scores = np.array(r[scores_key])
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linestyle=ls, lw=2, label=f"{label} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves")
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot4_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 5: Precision-Recall Curve
# ============================================================================
print("Plot 5: Precision-Recall Curve...")
fig, ax = plt.subplots(figsize=(8, 6))

# Use best hybrid model (best tier, best fold by F1)
best_tier_f1 = max(tier_names, key=lambda t: np.mean(
    [phase2_results[t][f]["hybrid_metrics"]["f1"] for f in phase2_results[t]]))
best_fold_pr = max(phase2_results[best_tier_f1].keys(),
                   key=lambda k: phase2_results[best_tier_f1][k]["hybrid_metrics"]["f1"])

r = phase2_results[best_tier_f1][best_fold_pr]
y_true = np.array(r["clip_labels"])
y_scores = np.array(r["clip_scores_xgb"])

precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_f1 = f1_scores[best_idx]
best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
best_prec = precision[best_idx]
best_rec = recall[best_idx]

ax.plot(recall, precision, color="#4CAF50", lw=2)
ax.scatter([best_rec], [best_prec], color="red", s=100, zorder=5)
ax.annotate(f"Best F1={best_f1:.3f}\nat threshold={best_thresh:.3f}",
            xy=(best_rec, best_prec),
            xytext=(best_rec - 0.2, best_prec - 0.1),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=10, color="red", fontweight="bold")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (Best Hybrid: {best_tier_f1}, Fold {best_fold_pr})")
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot5_pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 6: Confusion Matrix
# ============================================================================
print("Plot 6: Confusion Matrix...")
fig, ax = plt.subplots(figsize=(6, 5))

# Aggregate predictions across all folds for best tier hybrid
all_true = []
all_pred = []
for fold in phase2_results[best_tier_f1]:
    r = phase2_results[best_tier_f1][fold]
    for cr in r["clip_results"]:
        all_true.append(cr["true_label"])
        all_pred.append(cr["hybrid_pred"])

cm = confusion_matrix(all_true, all_pred)
total = cm.sum()

# Annotate with counts and percentages
annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm[i, j]}\n({cm[i, j] / total * 100:.1f}%)"

sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", ax=ax,
            xticklabels=["Normal", "Shoplifting"],
            yticklabels=["Normal", "Shoplifting"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (Hybrid {best_tier_f1}, All Folds)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot6_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# PLOT 7: Per-Fold Stability
# ============================================================================
print("Plot 7: Per-Fold Stability...")
fig, ax = plt.subplots(figsize=(8, 4))

fold_f1s = [phase2_results[best_tier_f1][f]["hybrid_metrics"]["f1"]
            for f in sorted(phase2_results[best_tier_f1].keys())]
folds = list(range(1, len(fold_f1s) + 1))
mean_f1 = np.mean(fold_f1s)

ax.bar(folds, fold_f1s, color="#4CAF50", edgecolor="black", alpha=0.8)
ax.axhline(y=mean_f1, color="red", linestyle="--", lw=2, label=f"Mean F1={mean_f1:.4f}")
for i, f1 in enumerate(fold_f1s):
    ax.text(folds[i], f1 + 0.005, f"{f1:.3f}", ha="center", fontsize=9)

ax.set_xlabel("Fold")
ax.set_ylabel("F1 Score")
ax.set_title(f"Per-Fold Stability (Hybrid {best_tier_f1})")
ax.set_xticks(folds)
ax.legend()
ax.set_ylim(0.5, 1.0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot7_fold_stability.png", dpi=150, bbox_inches="tight")
plt.show()

# ============================================================================
# ERROR ANALYSIS
# ============================================================================
print("\nError Analysis...")

lines = []
lines.append("=" * 60)
lines.append("ERROR ANALYSIS")
lines.append(f"Model: Hybrid {best_tier_f1}")
lines.append("=" * 60)

all_fp = []
all_fn = []

for fold in sorted(phase2_results[best_tier_f1].keys()):
    r = phase2_results[best_tier_f1][fold]
    for cr in r["clip_results"]:
        if cr["true_label"] == 0 and cr["hybrid_pred"] == 1:
            all_fp.append({"clip": cr["clip_name"], "score": cr["hybrid_score"], "fold": fold})
        elif cr["true_label"] == 1 and cr["hybrid_pred"] == 0:
            all_fn.append({"clip": cr["clip_name"], "score": cr["hybrid_score"], "fold": fold})

lines.append("\n=== FALSE POSITIVES (Normal -> predicted Shoplifting) ===")
for fp in sorted(all_fp, key=lambda x: -x["score"]):
    lines.append(f"  {fp['clip']} | score={fp['score']:.4f} | fold {fp['fold']}")

lines.append(f"\n=== FALSE NEGATIVES (Shoplifting -> predicted Normal) ===")
for fn in sorted(all_fn, key=lambda x: x["score"]):
    lines.append(f"  {fn['clip']} | score={fn['score']:.4f} | fold {fn['fold']}")

# Count unique clips
n_normal_clips = len(set(cr["clip_name"] for fold in phase2_results[best_tier_f1]
                         for cr in phase2_results[best_tier_f1][fold]["clip_results"]
                         if cr["true_label"] == 0))
n_shop_clips = len(set(cr["clip_name"] for fold in phase2_results[best_tier_f1]
                        for cr in phase2_results[best_tier_f1][fold]["clip_results"]
                        if cr["true_label"] == 1))

fp_unique = len(set(fp["clip"] for fp in all_fp))
fn_unique = len(set(fn["clip"] for fn in all_fn))

lines.append(f"\nSummary:")
lines.append(f"  FP: {fp_unique} unique normal clips misclassified across folds")
lines.append(f"  FN: {fn_unique} unique shoplifting clips misclassified across folds")

# Repeat offenders
from collections import Counter
fp_counter = Counter(fp["clip"] for fp in all_fp)
fn_counter = Counter(fn["clip"] for fn in all_fn)
repeat_fp = [clip for clip, count in fp_counter.items() if count >= 2]
repeat_fn = [clip for clip, count in fn_counter.items() if count >= 2]
lines.append(f"  Repeat FP offenders (misclassified 2+ times): {repeat_fp if repeat_fp else 'None'}")
lines.append(f"  Repeat FN offenders (misclassified 2+ times): {repeat_fn if repeat_fn else 'None'}")

error_text = "\n".join(lines)
print(error_text)

with open(RESULTS_DIR / "error_analysis.txt", "w") as f:
    f.write(error_text)

print("\nAll plots saved to v10_Results/")
print("Error analysis saved to v10_Results/error_analysis.txt")
print("Done.")


## 5. Error Analysis

Examining misclassified clips reveals where the model struggles and guides future improvement.
False positives (normal clips predicted as shoplifting) and false negatives (missed shoplifting)
are analyzed with their confidence scores to identify systematic failure patterns.

In [ ]:
# Error analysis is included in the visualization script above.
# Display the saved error analysis report:
with open(r"v10_Results/error_analysis.txt", "r") as f:
    print(f.read())

## 6. Limitations and Future Work

**Dataset limitations:**
- 358 clips is a relatively small dataset -- the model may not generalize to different store layouts,
  camera angles, or lighting conditions without additional training data.
- All clips come from a limited number of source videos, which may introduce scene-specific biases.

**Privacy considerations:**
- Pose estimation is significantly more privacy-preserving than raw video analysis, as only skeleton
  coordinates are stored and processed. However, the initial YOLO inference still requires access
  to raw frames during feature extraction.

**Feature tier analysis:**
- Tier C (spatial features) provided the best performance, suggesting that the spatial relationship
  between hands and body (wrist-to-hip distance, cross-body reach) captures the most discriminative
  signal for shoplifting behavior.
- Tier D (full features) showed signs of overfitting, with higher training metrics but lower
  validation F1 compared to Tier C. This is expected given the dataset size and the additional
  8 features adding noise without sufficient training examples.
- Velocity features (Tier B vs A) provided modest improvement, likely because frame-to-frame
  deltas are noisy with only 15-30 frames per track.

**Future directions:**
- **Architecture:** Spatio-temporal graph convolutional networks (ST-GCN) or Transformers with
  attention mechanisms could better capture joint relationships and long-range temporal dependencies.
- **Interpretability:** Attention visualization to identify which body parts and time steps
  contribute most to shoplifting predictions.
- **Deployment:** ONNX export for real-time edge inference on store security hardware.
- **Data:** Expand to multi-store, multi-camera datasets with diverse shoplifting techniques.